# Day 22 - math + random + logging

---

**What you will learn today:**

- math module: constants, arithmetic, logarithms, trigonometry, combinatorics
- random module: reproducibility, distributions, sampling
- random vs secrets: when to use which and why
- logging module: 5 levels, handlers, formatters, production patterns
- Monte Carlo Pi estimation: classic CS algorithm combining math and random
- Production ML training logger: the logging pattern used in real AI projects

---

**Why Standard Library matters:**

Every Python interview and every production codebase assumes you know the standard library. These are not third-party packages you have to install. They come with Python. They are fast, well-tested, and already available everywhere.

The three modules today are used in almost every serious Python project:
- math: scientific computing, geometry, probability calculations
- random: data augmentation, sampling, simulations, ML experiments
- logging: every production application uses logging instead of print()

---

# Part 1: The math Module

## Why math When We Have NumPy?

This is a question beginners often ask. The answer is:

NumPy is for arrays. math is for single numbers.

- `math.sqrt(9)` - fast for one number, returns a Python float
- `numpy.sqrt(array)` - fast for millions of numbers, returns an array

If you call `math.sqrt` on a NumPy array, it raises an error. If you call `numpy.sqrt` on a single number, it works but creates unnecessary overhead.

Use math when: you are working with individual scalar values, writing algorithms that operate number by number, or when you do not want to import NumPy (scripts, Lambda functions, environments with no NumPy).

Use NumPy when: you have arrays, DataFrames, vectorized operations.

---

## Mathematical Constants in Python

A constant is a value that never changes. The math module provides the most important mathematical constants as pre-computed, maximum-precision floating point numbers.

In Python, floating point numbers are 64-bit IEEE 754 doubles. They have about 15-17 significant decimal digits of precision. The constants in math module are stored at that maximum precision.

- `math.pi` = 3.141592653589793 (ratio of circumference to diameter of a circle)
- `math.e` = 2.718281828459045 (base of natural logarithm)
- `math.tau` = 6.283185307179586 (2 * pi, one full radian circle)
- `math.inf` = infinity (a float that is larger than any other float)
- `math.nan` = Not a Number (result of undefined operations like 0/0 or inf-inf)

inf and nan are special float values defined in the IEEE 754 standard. They are not errors - they are valid floating point values that represent special mathematical concepts.

In [1]:
import math
import random
import logging
import time

print('--- math Module Constants ---')
print(f'math.pi  = {math.pi}')
print(f'math.e   = {math.e}')
print(f'math.tau = {math.tau}   (= 2 * pi)')
print(f'math.inf = {math.inf}')
print(f'math.nan = {math.nan}')
print()

# Why tau exists: tau = 2*pi is the actual full circle in radians.
# A full circle is tau radians, not 2*pi radians (even though they are equal).
# Tau makes formulas cleaner: circumference = tau * r instead of 2*pi*r
print(f'math.tau == 2 * math.pi: {math.tau == 2 * math.pi}')
print()

# inf behavior
print('--- infinity behavior ---')
print(f'math.inf + 1000    = {math.inf + 1000}')    # still infinity
print(f'math.inf * -1      = {math.inf * -1}')       # negative infinity
print(f'math.inf - math.inf = {math.inf - math.inf}') # undefined = nan
print(f'1 / math.inf       = {1 / math.inf}')        # approaches 0
print()

# nan behavior - contagious
print('--- nan behavior ---')
print(f'math.nan + 5  = {math.nan + 5}')   # nan + anything = nan
print(f'math.nan == math.nan = {math.nan == math.nan}')  # FALSE! nan is not equal to itself
# This is why you CANNOT check for nan with ==. You must use math.isnan()
print(f'math.isnan(math.nan) = {math.isnan(math.nan)}')  # correct way

--- math Module Constants ---
math.pi  = 3.141592653589793
math.e   = 2.718281828459045
math.tau = 6.283185307179586   (= 2 * pi)
math.inf = inf
math.nan = nan

math.tau == 2 * math.pi: True

--- infinity behavior ---
math.inf + 1000    = inf
math.inf * -1      = -inf
math.inf - math.inf = nan
1 / math.inf       = 0.0

--- nan behavior ---
math.nan + 5  = nan
math.nan == math.nan = False
math.isnan(math.nan) = True


### The nan Equality Trap - Critical to Know

The fact that `nan != nan` is one of the most confusing things in floating point arithmetic for beginners.

The IEEE 754 standard defines nan as not equal to anything, including itself. The reasoning: nan means "the result of some undefined operation." Two different undefined operations are not necessarily the same undefined thing. So `nan == nan` is False by definition.

This is why:
- `float('nan') == float('nan')` is False
- `math.nan == math.nan` is False
- `numpy.nan == numpy.nan` is False
- In Pandas, `df['col'].isnull()` is the correct way to check for NaN, not `df['col'] == NaN`

Always use `math.isnan()`, `numpy.isnan()`, or `pandas.isnull()` to check for NaN.

In [2]:
print('--- Basic Operations ---')

# sqrt: square root
# Returns float always, even for perfect squares
print(f'math.sqrt(16)   = {math.sqrt(16)}')    # 4.0 not 4
print(f'math.sqrt(2)    = {math.sqrt(2)}')
print()

# pow: power (math.pow always returns float, ** operator returns int for int inputs)
print(f'math.pow(2, 10) = {math.pow(2, 10)}')  # 1024.0 (float)
print(f'2 ** 10         = {2 ** 10}')           # 1024 (int)
# math.pow is useful when you need float output consistently
print()

# fabs: absolute value that always returns float
# abs() is built-in and works on int/float/complex. fabs() is only for float.
print(f'math.fabs(-7.5) = {math.fabs(-7.5)}')  # 7.5
print(f'abs(-7.5)       = {abs(-7.5)}')         # 7.5 (same result but built-in)
print()

# factorial: n! = n * (n-1) * (n-2) * ... * 1
# Used in probability and combinatorics
print(f'math.factorial(5)  = {math.factorial(5)}')   # 120
print(f'math.factorial(10) = {math.factorial(10)}')  # 3628800
print(f'math.factorial(20) = {math.factorial(20)}')  # Python handles big integers
print()

# gcd and lcm: greatest common divisor and least common multiple
# gcd: largest number that divides both without remainder
# lcm: smallest number that both divide into without remainder
print(f'math.gcd(12, 8)    = {math.gcd(12, 8)}')    # 4
print(f'math.lcm(4, 6)     = {math.lcm(4, 6)}')     # 12
print(f'math.gcd(100, 75)  = {math.gcd(100, 75)}')  # 25
print()

# floor, ceil, trunc: three different ways to convert float to integer
x = 3.7
y = -3.7
print(f'Value: {x}')
print(f'math.floor({x})  = {math.floor(x)}   (round DOWN to nearest integer)')
print(f'math.ceil({x})   = {math.ceil(x)}    (round UP to nearest integer)')
print(f'math.trunc({x})  = {math.trunc(x)}   (truncate decimal, toward zero)')
print()
print(f'Value: {y}')
print(f'math.floor({y}) = {math.floor(y)}   (round DOWN: -4, not -3)')
print(f'math.ceil({y})  = {math.ceil(y)}    (round UP: -3, not -4)')
print(f'math.trunc({y}) = {math.trunc(y)}   (truncate toward zero: -3)')
print()
print('IMPORTANT: floor(-3.7) = -4 because floor always goes DOWN (more negative)')
print('trunc(-3.7) = -3 because trunc always goes TOWARD ZERO')

--- Basic Operations ---
math.sqrt(16)   = 4.0
math.sqrt(2)    = 1.4142135623730951

math.pow(2, 10) = 1024.0
2 ** 10         = 1024

math.fabs(-7.5) = 7.5
abs(-7.5)       = 7.5

math.factorial(5)  = 120
math.factorial(10) = 3628800
math.factorial(20) = 2432902008176640000

math.gcd(12, 8)    = 4
math.lcm(4, 6)     = 12
math.gcd(100, 75)  = 25

Value: 3.7
math.floor(3.7)  = 3   (round DOWN to nearest integer)
math.ceil(3.7)   = 4    (round UP to nearest integer)
math.trunc(3.7)  = 3   (truncate decimal, toward zero)

Value: -3.7
math.floor(-3.7) = -4   (round DOWN: -4, not -3)
math.ceil(-3.7)  = -3    (round UP: -3, not -4)
math.trunc(-3.7) = -3   (truncate toward zero: -3)

IMPORTANT: floor(-3.7) = -4 because floor always goes DOWN (more negative)
trunc(-3.7) = -3 because trunc always goes TOWARD ZERO


### floor vs ceil vs trunc vs round - The Four Ways to Remove Decimals

This is a source of bugs in many programs. Understanding the difference is essential.

Given x = 3.7 and y = -3.7:

```
Function     x=3.7    y=-3.7    Rule
floor()        3        -4      always toward negative infinity
ceil()         4        -3      always toward positive infinity
trunc()        3        -3      always toward zero
round()        4        -4      toward nearest, ties to even
int()          3        -3      same as trunc()
```

For positive numbers, floor = trunc = int. They differ only for negative numbers.

In financial calculations: use floor for fees (do not overcharge), use ceil for minimums (do not undercharge).

In [3]:
print('--- Logarithms and Exponentials ---')
print()

# What is a logarithm?
# log_b(x) = y means b^y = x
# "How many times do I multiply b by itself to get x?"

# math.log(x, base): logarithm of x to the given base
# math.log(x): natural logarithm (base e)
print('math.log(x, base): logarithm base b of x')
print(f'math.log(100, 10) = {math.log(100, 10)}  (because 10^2 = 100)')
print(f'math.log(8, 2)    = {math.log(8, 2)}     (because 2^3 = 8)')
print(f'math.log(math.e)  = {math.log(math.e)}   (natural log of e = 1)')
print()

# Specialized log functions - use these instead of math.log(x, 2) for accuracy
print('Specialized log functions (more accurate for specific bases):')
print(f'math.log2(8)      = {math.log2(8)}    (log base 2 - bits, information theory)')
print(f'math.log10(1000)  = {math.log10(1000)} (log base 10 - decibels, pH, Richter scale)')
print()

# Exponential functions
# math.exp(x) = e^x
print('math.exp(x) = e raised to power x:')
print(f'math.exp(1) = {math.exp(1)}')  # e^1 = e
print(f'math.exp(0) = {math.exp(0)}')  # e^0 = 1
print(f'math.exp(2) = {math.exp(2)}')
print()

# log1p and expm1: numerically stable versions for values near zero
# This is advanced but important for ML loss functions and probability calculations
print('log1p and expm1: numerically stable for small x')
x_small = 1e-15  # very small number
print(f'x = {x_small}')
print(f'math.log(1 + x_small) = {math.log(1 + x_small)} (loses precision due to 1+x)')
print(f'math.log1p(x_small)   = {math.log1p(x_small)} (more accurate for small x)')
print()
print('Why this matters: in softmax, cross-entropy, and probability calculations,')
print('values near 0 are common. log1p avoids catastrophic cancellation.')

--- Logarithms and Exponentials ---

math.log(x, base): logarithm base b of x
math.log(100, 10) = 2.0  (because 10^2 = 100)
math.log(8, 2)    = 3.0     (because 2^3 = 8)
math.log(math.e)  = 1.0   (natural log of e = 1)

Specialized log functions (more accurate for specific bases):
math.log2(8)      = 3.0    (log base 2 - bits, information theory)
math.log10(1000)  = 3.0 (log base 10 - decibels, pH, Richter scale)

math.exp(x) = e raised to power x:
math.exp(1) = 2.718281828459045
math.exp(0) = 1.0
math.exp(2) = 7.38905609893065

log1p and expm1: numerically stable for small x
x = 1e-15
math.log(1 + x_small) = 1.110223024625156e-15 (loses precision due to 1+x)
math.log1p(x_small)   = 9.999999999999997e-16 (more accurate for small x)

Why this matters: in softmax, cross-entropy, and probability calculations,
values near 0 are common. log1p avoids catastrophic cancellation.


In [4]:
print('--- Trigonometry ---')
print()

# IMPORTANT: All trig functions in math use RADIANS, not degrees
# Radians and degrees are two systems to measure angles:
# - Degrees: 0 to 360 for a full circle (everyday language)
# - Radians: 0 to 2*pi for a full circle (mathematics and programming)
# Conversion: radians = degrees * pi / 180

print('Angles: math uses RADIANS, not degrees')
print(f'math.radians(180) = {math.radians(180):.6f}  (pi radians = 180 degrees)')
print(f'math.degrees(math.pi) = {math.degrees(math.pi)}')
print()

# Convert common angles
for deg in [0, 30, 45, 60, 90, 180, 270, 360]:
    rad = math.radians(deg)
    print(f'{deg:3d} degrees = {rad:.4f} radians')
print()

# sin, cos, tan
print('sin, cos, tan of common angles:')
print(f'sin(0)    = {math.sin(0):.4f}')
print(f'sin(pi/2) = {math.sin(math.pi/2):.4f}  (sin of 90 degrees = 1)')
print(f'cos(0)    = {math.cos(0):.4f}  (cos of 0 = 1)')
print(f'cos(pi)   = {math.cos(math.pi):.4f} (cos of 180 degrees = -1)')
print(f'tan(pi/4) = {math.tan(math.pi/4):.4f}  (tan of 45 degrees = 1)')
print()

# Floating point precision issue with trig
print('Floating point precision note:')
print(f'math.sin(math.pi) = {math.sin(math.pi)}  (should be 0 but floating point error)')
print('This is not a Python bug. It is IEEE 754 floating point limitation.')
print(f'The error is {math.sin(math.pi):.2e} which is negligibly small.')
print()

# hypot: hypotenuse of a right triangle (Pythagorean theorem)
# hypot(x, y) = sqrt(x^2 + y^2)
# More numerically stable than sqrt(x**2 + y**2)
print('math.hypot: Pythagorean distance')
print(f'hypot(3, 4) = {math.hypot(3, 4)}   (classic 3-4-5 triangle)')
print(f'hypot(5, 12) = {math.hypot(5, 12)}  (5-12-13 triangle)')
# In Python 3.8+, hypot accepts more than 2 arguments (Euclidean distance in N dimensions)
print(f'hypot(1, 2, 2) = {math.hypot(1, 2, 2)}  (3D distance: sqrt(1+4+4) = 3)')
print()

# atan2: angle of a point (x, y) from origin
# atan2(y, x) is preferred over atan(y/x) because it handles all quadrants correctly
print('atan2 vs atan: atan2 handles all quadrants')
print(f'atan2(1, 1)   = {math.degrees(math.atan2(1, 1)):.1f} degrees  (45 degrees, first quadrant)')
print(f'atan2(1, -1)  = {math.degrees(math.atan2(1, -1)):.1f} degrees (135 degrees, second quadrant)')
print(f'atan2(-1, -1) = {math.degrees(math.atan2(-1, -1)):.1f} degrees (225/-135, third quadrant)')

--- Trigonometry ---

Angles: math uses RADIANS, not degrees
math.radians(180) = 3.141593  (pi radians = 180 degrees)
math.degrees(math.pi) = 180.0

  0 degrees = 0.0000 radians
 30 degrees = 0.5236 radians
 45 degrees = 0.7854 radians
 60 degrees = 1.0472 radians
 90 degrees = 1.5708 radians
180 degrees = 3.1416 radians
270 degrees = 4.7124 radians
360 degrees = 6.2832 radians

sin, cos, tan of common angles:
sin(0)    = 0.0000
sin(pi/2) = 1.0000  (sin of 90 degrees = 1)
cos(0)    = 1.0000  (cos of 0 = 1)
cos(pi)   = -1.0000 (cos of 180 degrees = -1)
tan(pi/4) = 1.0000  (tan of 45 degrees = 1)

Floating point precision note:
math.sin(math.pi) = 1.2246467991473532e-16  (should be 0 but floating point error)
This is not a Python bug. It is IEEE 754 floating point limitation.
The error is 1.22e-16 which is negligibly small.

math.hypot: Pythagorean distance
hypot(3, 4) = 5.0   (classic 3-4-5 triangle)
hypot(5, 12) = 13.0  (5-12-13 triangle)
hypot(1, 2, 2) = 3.0  (3D distance: sqrt(1+4+4)

In [5]:
print('--- Comparison and Validation Functions ---')
print()

# isclose: floating point comparison
# NEVER compare floats with == because of floating point errors
# Always use isclose for float comparison

print('CRITICAL: Never use == for floating point comparison')
a = 0.1 + 0.2
b = 0.3
print(f'0.1 + 0.2 = {a}')
print(f'0.3       = {b}')
print(f'0.1 + 0.2 == 0.3: {a == b}')  # FALSE due to floating point
print(f'math.isclose(0.1+0.2, 0.3): {math.isclose(a, b)}')  # TRUE - correct way
print()

# isclose parameters:
# rel_tol: relative tolerance (default 1e-9). Scales with the magnitude of the numbers.
# abs_tol: absolute tolerance (default 0). Use when comparing to zero.
print('isclose parameters:')
print(f'isclose(1000.1, 1000.2, rel_tol=1e-3): {math.isclose(1000.1, 1000.2, rel_tol=1e-3)}')
print(f'isclose(0.0, 1e-10, abs_tol=1e-9): {math.isclose(0.0, 1e-10, abs_tol=1e-9)}')
print()

# isinf, isnan, isfinite: checking special float values
print('Checking special float values:')
values = [0.0, 1.5, math.inf, -math.inf, math.nan, 1e308]
for v in values:
    print(f'  {str(v):10s} | isfinite={math.isfinite(v)} | isinf={math.isinf(v)} | isnan={math.isnan(v)}')
print()

# fmod: floating point modulo
# Unlike %, fmod gives result with same sign as dividend (x), not divisor
print('fmod vs % for negative numbers:')
print(f'math.fmod(-7, 3) = {math.fmod(-7, 3)}   (sign of -7: negative)')
print(f'-7 % 3           = {-7 % 3}    (sign of 3: positive - Python behavior)')
print('C/C++ behavior matches fmod. Python % behavior is different for negatives.')

--- Comparison and Validation Functions ---

CRITICAL: Never use == for floating point comparison
0.1 + 0.2 = 0.30000000000000004
0.3       = 0.3
0.1 + 0.2 == 0.3: False
math.isclose(0.1+0.2, 0.3): True

isclose parameters:
isclose(1000.1, 1000.2, rel_tol=1e-3): True
isclose(0.0, 1e-10, abs_tol=1e-9): True

Checking special float values:
  0.0        | isfinite=True | isinf=False | isnan=False
  1.5        | isfinite=True | isinf=False | isnan=False
  inf        | isfinite=False | isinf=True | isnan=False
  -inf       | isfinite=False | isinf=True | isnan=False
  nan        | isfinite=False | isinf=False | isnan=True
  1e+308     | isfinite=True | isinf=False | isnan=False

fmod vs % for negative numbers:
math.fmod(-7, 3) = -1.0   (sign of -7: negative)
-7 % 3           = 2    (sign of 3: positive - Python behavior)
C/C++ behavior matches fmod. Python % behavior is different for negatives.


In [6]:
print('--- Combinatorics and Special Functions ---')
print()

# comb: combinations - C(n, k) = n! / (k! * (n-k)!)
# How many ways to choose k items from n items (order does NOT matter)
# Example: how many ways to choose 3 students from a class of 10?
print('math.comb(n, k): combinations (order does not matter)')
print(f'comb(10, 3) = {math.comb(10, 3)}  (10 choose 3: teams of 3 from 10 people)')
print(f'comb(52, 5) = {math.comb(52, 5)}  (5-card poker hands from 52-card deck)')
print(f'comb(6, 6)  = {math.comb(6, 6)}   (choosing all = 1 way)')
print(f'comb(6, 0)  = {math.comb(6, 0)}   (choosing none = 1 way)')
print()

# perm: permutations - P(n, k) = n! / (n-k)!
# How many ways to arrange k items from n items (order MATTERS)
# Example: how many ways can 3 students finish first, second, third from 10?
print('math.perm(n, k): permutations (order matters)')
print(f'perm(10, 3) = {math.perm(10, 3)}   (top-3 finishers from 10: order matters)')
print(f'perm(4, 4)  = {math.perm(4, 4)}    (arrangements of all 4: = 4! = 24)')
print()
print('comb vs perm: choosing 2 from [A, B, C]')
print(f'comb(3,2) = {math.comb(3,2)}: [AB, AC, BC] (order does not matter, AB = BA)')
print(f'perm(3,2) = {math.perm(3,2)}: [AB, BA, AC, CA, BC, CB] (AB != BA)')
print()

# prod: product of all items in an iterable
# Like sum() but multiplication
print('math.prod: multiply all elements together')
print(f'prod([1,2,3,4,5]) = {math.prod([1,2,3,4,5])}   (same as factorial(5))')
print(f'prod([2,4,8,16])  = {math.prod([2,4,8,16])}')
print()

# fsum: accurate floating point sum
# Regular sum() accumulates floating point errors. fsum() is more accurate.
print('math.fsum vs sum: accurate floating point summation')
nums = [0.1] * 10  # ten 0.1 values, should sum to 1.0
print(f'nums = [0.1] * 10')
print(f'sum(nums)  = {sum(nums)}   (floating point accumulation error)')
print(f'fsum(nums) = {math.fsum(nums)}  (exact result)')

--- Combinatorics and Special Functions ---

math.comb(n, k): combinations (order does not matter)
comb(10, 3) = 120  (10 choose 3: teams of 3 from 10 people)
comb(52, 5) = 2598960  (5-card poker hands from 52-card deck)
comb(6, 6)  = 1   (choosing all = 1 way)
comb(6, 0)  = 1   (choosing none = 1 way)

math.perm(n, k): permutations (order matters)
perm(10, 3) = 720   (top-3 finishers from 10: order matters)
perm(4, 4)  = 24    (arrangements of all 4: = 4! = 24)

comb vs perm: choosing 2 from [A, B, C]
comb(3,2) = 3: [AB, AC, BC] (order does not matter, AB = BA)
perm(3,2) = 6: [AB, BA, AC, CA, BC, CB] (AB != BA)

math.prod: multiply all elements together
prod([1,2,3,4,5]) = 120   (same as factorial(5))
prod([2,4,8,16])  = 1024

math.fsum vs sum: accurate floating point summation
nums = [0.1] * 10
sum(nums)  = 1.0   (floating point accumulation error)
fsum(nums) = 1.0  (exact result)


---

# Part 2: The random Module

## How random Actually Works

The random module does NOT generate truly random numbers. It generates pseudo-random numbers using the Mersenne Twister algorithm.

Mersenne Twister is a deterministic algorithm that produces a sequence of numbers that appear random but are mathematically computed from a starting value called the seed.

Given the same seed, the algorithm always produces the exact same sequence of numbers. This is not a bug. It is a feature called reproducibility.

### Why Reproducibility Matters in ML

When you train a neural network:
- Weights are initialized randomly
- Data is shuffled randomly
- Dropout randomly drops neurons

If you cannot reproduce the random state, you cannot:
- Debug a failed training run
- Share your exact results with another researcher
- Verify that a model update actually improved results (or just got lucky)

So every ML project starts with:
```python
random.seed(42)
numpy.random.seed(42)
torch.manual_seed(42)  # if using PyTorch
```

The number 42 is convention (from The Hitchhiker's Guide to the Galaxy) but any integer works. The point is consistency.

---

## The Difference Between random's Four Basic Functions

This confuses beginners. Here is the clear breakdown:

- `random.random()` - float in [0.0, 1.0). No arguments. The fundamental function.
- `random.uniform(a, b)` - float in [a, b]. Both endpoints included.
- `random.randint(a, b)` - integer in [a, b]. BOTH endpoints included (unusual for Python).
- `random.randrange(start, stop, step)` - integer in [start, stop). Stop NOT included. Like range().

In [7]:
print('--- random Module: Reproducibility ---')
print()

# Without seed: different numbers every run
print('Without seed (different every run):')
print([random.random() for _ in range(5)])
print([random.random() for _ in range(5)])
print()

# With seed: same sequence every time
print('With seed(42) (same every run):')
random.seed(42)
run1 = [random.random() for _ in range(5)]
print(f'Run 1: {run1}')

random.seed(42)  # reset to same seed
run2 = [random.random() for _ in range(5)]
print(f'Run 2: {run2}')

print(f'Identical: {run1 == run2}')
print()
print('This is why ML experiments use fixed seeds:')
print('Same seed = same "randomness" = reproducible results.')

--- random Module: Reproducibility ---

Without seed (different every run):
[0.3284386785728831, 0.2602050030240991, 0.7626401546600019, 0.8284233445774827, 0.7220922813112561]
[0.43726673589450515, 0.1900974888501552, 0.7640472482344044, 0.5599375355008102, 0.06932093198695233]

With seed(42) (same every run):
Run 1: [0.6394267984578837, 0.025010755222666936, 0.27502931836911926, 0.22321073814882275, 0.7364712141640124]
Run 2: [0.6394267984578837, 0.025010755222666936, 0.27502931836911926, 0.22321073814882275, 0.7364712141640124]
Identical: True

This is why ML experiments use fixed seeds:
Same seed = same "randomness" = reproducible results.


In [8]:
print('--- Four Basic random Functions ---')
print()

random.seed(42)

# random.random(): float in [0.0, 1.0)
# [0.0, 1.0) means 0.0 is possible but 1.0 is never returned
# This is the CORE function. All others are built on this.
print('random.random(): float in [0.0, 1.0)')
print([round(random.random(), 4) for _ in range(6)])
print()

# random.uniform(a, b): float in [a, b]
# Note: unlike random(), both endpoints CAN be returned (though unlikely)
print('random.uniform(a, b): float in [a, b]')
print(f'uniform(5, 10): {[round(random.uniform(5, 10), 2) for _ in range(6)]}')
print(f'uniform(-1, 1): {[round(random.uniform(-1, 1), 4) for _ in range(6)]}')
print()

# random.randint(a, b): integer in [a, b] - BOTH ENDPOINTS INCLUDED
# This is DIFFERENT from range(a, b) where b is excluded
# randint(1, 6) simulates a 6-sided die (1, 2, 3, 4, 5, or 6)
print('random.randint(a, b): integer in [a, b] - BOTH included')
dice_rolls = [random.randint(1, 6) for _ in range(10)]
print(f'10 dice rolls: {dice_rolls}')
print()

# random.randrange(start, stop, step): like range() but random
# Stop is NOT included (same as range)
print('random.randrange(start, stop, step): integer in [start, stop) like range()')
print(f'randrange(0, 10):    {[random.randrange(0, 10) for _ in range(6)]}')
print(f'randrange(0, 100, 5): {[random.randrange(0, 100, 5) for _ in range(6)]}  (multiples of 5)')
print(f'randrange(1, 2):      {[random.randrange(1, 2) for _ in range(5)]}  (always 1, stop not included)')

--- Four Basic random Functions ---

random.random(): float in [0.0, 1.0)
[0.6394, 0.025, 0.275, 0.2232, 0.7365, 0.6767]

random.uniform(a, b): float in [a, b]
uniform(5, 10): [9.46, 5.43, 7.11, 5.15, 6.09, 7.53]
uniform(-1, 1): [-0.9469, -0.6023, 0.2998, 0.0899, -0.5591, 0.1785]

random.randint(a, b): integer in [a, b] - BOTH included
10 dice rolls: [1, 2, 6, 4, 3, 3, 2, 2, 3, 1]

random.randrange(start, stop, step): integer in [start, stop) like range()
randrange(0, 10):    [1, 6, 1, 5, 5, 9]
randrange(0, 100, 5): [40, 5, 70, 85, 15, 60]  (multiples of 5)
randrange(1, 2):      [1, 1, 1, 1, 1]  (always 1, stop not included)


In [9]:
print('--- Probability Distributions ---')
print()

# random.gauss(mu, sigma): Gaussian (normal) distribution
# Most natural phenomena follow this bell curve:
# heights, weights, test scores, measurement errors
# mu = mean (center of the bell curve)
# sigma = standard deviation (width of the bell curve)
# About 68% of values fall within 1 sigma of mu
# About 95% within 2 sigma
# About 99.7% within 3 sigma

print('random.gauss(mu, sigma): Gaussian/Normal distribution')
print('Simulating IQ scores: mean=100, std=15')
random.seed(42)
iq_scores = [random.gauss(100, 15) for _ in range(10000)]
mean_iq = sum(iq_scores) / len(iq_scores)
within_1sigma = sum(1 for x in iq_scores if 85 <= x <= 115) / len(iq_scores)
within_2sigma = sum(1 for x in iq_scores if 70 <= x <= 130) / len(iq_scores)
print(f'  Sample mean: {mean_iq:.2f} (expected: 100)')
print(f'  Within 1 sigma (85-115): {within_1sigma:.1%} (expected: ~68%)')
print(f'  Within 2 sigma (70-130): {within_2sigma:.1%} (expected: ~95%)')
print()

# random.normalvariate(mu, sigma): same as gauss but thread-safe
# gauss is slightly faster but not thread-safe
# In multi-threaded applications, use normalvariate
print('random.normalvariate: same as gauss but thread-safe')
print('Use normalvariate in multi-threaded applications (FastAPI, concurrent.futures)')
print()

# random.expovariate(lambd): exponential distribution
# Models: time between random events (customer arrivals, server requests)
# lambd = 1/mean (rate parameter). If average = 5 minutes, lambd = 1/5 = 0.2
print('random.expovariate(lambd): exponential distribution')
print('Models time between events (customer arrivals, API requests)')
lambd = 0.2  # 1/mean = 1/5, so average interval = 5 units
wait_times = [random.expovariate(lambd) for _ in range(10000)]
mean_wait = sum(wait_times) / len(wait_times)
print(f'  lambd=0.2 -> expected mean = 5.0, sample mean = {mean_wait:.2f}')

--- Probability Distributions ---

random.gauss(mu, sigma): Gaussian/Normal distribution
Simulating IQ scores: mean=100, std=15
  Sample mean: 99.82 (expected: 100)
  Within 1 sigma (85-115): 68.3% (expected: ~68%)
  Within 2 sigma (70-130): 95.5% (expected: ~95%)

random.normalvariate: same as gauss but thread-safe
Use normalvariate in multi-threaded applications (FastAPI, concurrent.futures)

random.expovariate(lambd): exponential distribution
Models time between events (customer arrivals, API requests)
  lambd=0.2 -> expected mean = 5.0, sample mean = 5.03


In [10]:
print('--- Sampling Functions ---')
print()

fruits = ['apple', 'banana', 'cherry', 'date', 'elderberry', 'fig', 'grape']
random.seed(42)

# random.choice(seq): pick ONE random element from a sequence
# Equal probability for each element
print('random.choice(seq): pick one random element')
print(f'choice(fruits): {random.choice(fruits)}')
print(f'choice(fruits): {random.choice(fruits)}')
print(f'choice(fruits): {random.choice(fruits)}')
print()

# random.choices(seq, weights=, k=): pick k elements WITH replacement
# WITH replacement means the same element can be chosen multiple times
# weights= makes some elements more likely than others
print('random.choices(seq, weights=None, k=1): pick k WITH replacement')
print(f'choices(fruits, k=3): {random.choices(fruits, k=3)}  (may have duplicates)')

# Weighted: make apple 5x more likely than others
weights = [5, 1, 1, 1, 1, 1, 1]  # apple has weight 5, others have weight 1
print(f'choices with weights (apple=5x): {random.choices(fruits, weights=weights, k=10)}')
print()

# random.sample(seq, k): pick k elements WITHOUT replacement
# WITHOUT replacement means no duplicates - each element can only be chosen once
# k must be <= len(seq)
print('random.sample(seq, k): pick k WITHOUT replacement (no duplicates)')
print(f'sample(fruits, 3): {random.sample(fruits, 3)}')
print(f'sample(fruits, 5): {random.sample(fruits, 5)}')
print()

# choices vs sample: critical difference
print('choices vs sample: the key difference')
print('choices(population, k=3): can repeat - [apple, apple, grape] possible')
print('sample(population, 3): no repeats - [apple, banana, grape] guaranteed unique')
print()
print('Use cases:')
print('choices: simulating dice rolls, lottery with replacement, bootstrap sampling')
print('sample:  splitting train/test data, picking survey participants, drawing cards')
print()

# random.shuffle(seq): shuffle in-place - MODIFIES the original list
print('random.shuffle(seq): shuffle list IN-PLACE (modifies original)')
deck = list(range(1, 11))
print(f'Before shuffle: {deck}')
random.shuffle(deck)
print(f'After shuffle:  {deck}')
print()
print('NOTE: shuffle returns None. It modifies the list in-place.')
print('Wrong: shuffled = random.shuffle(deck)  <- shuffled will be None')
print('Right: random.shuffle(deck); then use deck')

--- Sampling Functions ---

random.choice(seq): pick one random element
choice(fruits): fig
choice(fruits): apple
choice(fruits): apple

random.choices(seq, weights=None, k=1): pick k WITH replacement
choices(fruits, k=3): ['fig', 'banana', 'apple']  (may have duplicates)
choices with weights (apple=5x): ['apple', 'elderberry', 'banana', 'cherry', 'apple', 'apple', 'apple', 'cherry', 'cherry', 'date']

random.sample(seq, k): pick k WITHOUT replacement (no duplicates)
sample(fruits, 3): ['fig', 'elderberry', 'date']
sample(fruits, 5): ['banana', 'date', 'elderberry', 'cherry', 'apple']

choices vs sample: the key difference
choices(population, k=3): can repeat - [apple, apple, grape] possible
sample(population, 3): no repeats - [apple, banana, grape] guaranteed unique

Use cases:
choices: simulating dice rolls, lottery with replacement, bootstrap sampling
sample:  splitting train/test data, picking survey participants, drawing cards

random.shuffle(seq): shuffle list IN-PLACE (modifies 

In [11]:
print('--- random vs secrets: When to Use Which ---')
print()

import secrets

# The core difference:
# random: uses Mersenne Twister (pseudo-random, deterministic, fast)
# secrets: uses OS-provided cryptographically secure random source
#
# random is predictable given the seed. If an attacker knows the seed,
# they can predict all future random numbers. This is FATAL for security.
#
# secrets uses /dev/urandom (Linux/Mac) or CryptGenRandom (Windows).
# These are based on physical entropy (hardware events, timing jitter).
# They are NOT predictable even with access to the source code.

print('random module: for ML, simulations, games, statistics')
print('  - Pseudo-random (Mersenne Twister algorithm)')
print('  - Deterministic (same seed = same output)')
print('  - Fast')
print('  - NOT suitable for security use cases')
print()
print('secrets module: for passwords, tokens, authentication')
print('  - Cryptographically secure (uses OS entropy source)')
print('  - Non-deterministic (cannot reproduce sequence)')
print('  - Slightly slower but negligible in practice')
print('  - Use for anything security-related')
print()

# secrets examples
print('secrets examples:')
print(f'secrets.token_hex(16):  {secrets.token_hex(16)}    (32 hex chars = 128 bits)')
print(f'secrets.token_urlsafe(16): {secrets.token_urlsafe(16)}  (URL-safe base64)')
print(f'secrets.randbelow(100): {secrets.randbelow(100)}  (secure random int 0-99)')
print()

# Practical rule:
print('THE RULE:')
print('If the number will be seen by a user (password, token, OTP) -> secrets')
print('If the number is internal (ML, simulation, games, stats)    -> random')
print()
print('Real examples:')
print('  API key generation          -> secrets.token_urlsafe()')
print('  Password reset token        -> secrets.token_hex()')
print('  ML train/test split         -> random.sample()')
print('  Shuffling training data     -> random.shuffle()')
print('  Monte Carlo simulation      -> random.random()')
print('  OTP (one-time password)     -> secrets.randbelow(1000000)')

--- random vs secrets: When to Use Which ---

random module: for ML, simulations, games, statistics
  - Pseudo-random (Mersenne Twister algorithm)
  - Deterministic (same seed = same output)
  - Fast
  - NOT suitable for security use cases

secrets module: for passwords, tokens, authentication
  - Cryptographically secure (uses OS entropy source)
  - Non-deterministic (cannot reproduce sequence)
  - Slightly slower but negligible in practice
  - Use for anything security-related

secrets examples:
secrets.token_hex(16):  1a333dd1af0c197a70d8facf76c9c5d9    (32 hex chars = 128 bits)
secrets.token_urlsafe(16): QvA3RtD08Bgy1tYjl4X1hw  (URL-safe base64)
secrets.randbelow(100): 85  (secure random int 0-99)

THE RULE:
If the number will be seen by a user (password, token, OTP) -> secrets
If the number is internal (ML, simulation, games, stats)    -> random

Real examples:
  API key generation          -> secrets.token_urlsafe()
  Password reset token        -> secrets.token_hex()
  ML train/

---

# Part 3: Monte Carlo Pi Estimation

## What is Monte Carlo Simulation?

Monte Carlo methods use random sampling to estimate mathematical values that would be hard to compute directly. The name comes from the famous Monte Carlo casino in Monaco.

The core idea: throw many random points into a space, count what fraction lands in a region of interest, and use that fraction to estimate geometric or probabilistic quantities.

## Why Pi Estimation is the Classic Example

Consider a circle with radius 1 inscribed in a square with side length 2:
- Area of the square = 2 * 2 = 4
- Area of the circle = pi * r^2 = pi * 1^2 = pi
- Ratio = pi / 4

If you throw random points uniformly inside the square, the fraction that lands inside the circle should be approximately pi/4.

A point (x, y) is inside the circle if x^2 + y^2 <= 1.

So: pi / 4 ≈ (points inside circle) / (total points)
Therefore: pi ≈ 4 * (points inside circle) / (total points)

## Why This Matters for Learning

This example is asked in interviews and taught in courses because it demonstrates:
1. How to use randomness to solve deterministic problems
2. The law of large numbers: more samples = better estimate
3. The sqrt connection: distance formula uses math.sqrt or math.hypot
4. Performance measurement with time.perf_counter
5. The relationship between probability and geometry

In [12]:
# MONTE CARLO PI ESTIMATION
# Shows: random + math + time.perf_counter together

def estimate_pi(n_points, seed=42):
    """
    Estimate pi using Monte Carlo method.
    
    Theory:
    - Throw n_points randomly into the square [-1,1] x [-1,1]
    - Count how many land inside unit circle: x^2 + y^2 <= 1
    - pi/4 = (points inside circle) / (total points)
    - Therefore pi ≈ 4 * inside / total
    """
    random.seed(seed)
    inside = 0
    
    for _ in range(n_points):
        # Random point in [-1, 1] x [-1, 1]
        x = random.uniform(-1, 1)
        y = random.uniform(-1, 1)
        
        # Check if inside unit circle using distance from origin
        # math.hypot(x, y) = sqrt(x^2 + y^2)
        # If distance <= 1, point is inside circle
        if math.hypot(x, y) <= 1.0:
            inside += 1
    
    pi_estimate = 4 * inside / n_points
    return pi_estimate

# Test with different sample sizes and measure time
print('Monte Carlo Pi Estimation')
print(f'True value of pi: {math.pi}')
print()
print(f'{"N points":>15}  {"Estimate":>12}  {"Error":>12}  {"Time (sec)":>12}')
print('-' * 60)

for n in [100, 1_000, 10_000, 100_000, 1_000_000]:
    start = time.perf_counter()   # accurate timing - prefer over time.time()
    estimate = estimate_pi(n)
    elapsed = time.perf_counter() - start
    
    error = abs(estimate - math.pi)
    print(f'{n:>15,}  {estimate:>12.6f}  {error:>12.6f}  {elapsed:>12.4f}')

print()
print('Observation: Error reduces by roughly 10x for every 100x increase in points.')
print('This is the square root law: accuracy improves with sqrt(N).')
print('To halve the error, you need 4x more points. To get 10x better, 100x more points.')

Monte Carlo Pi Estimation
True value of pi: 3.141592653589793

       N points      Estimate         Error    Time (sec)
------------------------------------------------------------
            100      3.200000      0.058407        0.0001
          1,000      3.180000      0.038407        0.0005
         10,000      3.139200      0.002393        0.0047
        100,000      3.140280      0.001313        0.0585
      1,000,000      3.140244      0.001349        0.4590

Observation: Error reduces by roughly 10x for every 100x increase in points.
This is the square root law: accuracy improves with sqrt(N).
To halve the error, you need 4x more points. To get 10x better, 100x more points.


In [13]:
# OPTIMIZED VERSION: Using list comprehension instead of for loop
# Demonstrates Python optimization techniques

def estimate_pi_fast(n_points, seed=42):
    """
    Optimized Monte Carlo Pi estimation.
    Uses list comprehension to avoid loop overhead.
    """
    random.seed(seed)
    
    # Generate all points at once and count inside in one pass
    inside = sum(
        1 for _ in range(n_points)
        if math.hypot(random.uniform(-1, 1), random.uniform(-1, 1)) <= 1
    )
    
    return 4 * inside / n_points

print('Performance comparison: loop vs generator expression')
n = 100_000

start = time.perf_counter()
pi1 = estimate_pi(n)
t1 = time.perf_counter() - start

start = time.perf_counter()
pi2 = estimate_pi_fast(n)
t2 = time.perf_counter() - start

print(f'Loop version:      pi={pi1:.6f}, time={t1:.4f}s')
print(f'Generator version: pi={pi2:.6f}, time={t2:.4f}s')
print()
print('For serious performance with large N, use NumPy:')
print('  import numpy as np')
print('  xy = np.random.uniform(-1, 1, (n, 2))')
print('  inside = (xy[:,0]**2 + xy[:,1]**2 <= 1).sum()')
print('  pi_estimate = 4 * inside / n')
print('NumPy version is 50-100x faster due to vectorization.')

Performance comparison: loop vs generator expression
Loop version:      pi=3.140280, time=0.0442s
Generator version: pi=3.140280, time=0.0470s

For serious performance with large N, use NumPy:
  import numpy as np
  xy = np.random.uniform(-1, 1, (n, 2))
  inside = (xy[:,0]**2 + xy[:,1]**2 <= 1).sum()
  pi_estimate = 4 * inside / n
NumPy version is 50-100x faster due to vectorization.


---

# Part 4: The logging Module

## Why Logging Instead of print()

This is one of the most important professional habits in Python development.

**Problems with print():**

1. Cannot be turned off without deleting code. In production, you do not want debug messages visible. With print(), you must delete them or comment them out.

2. No timestamp. You cannot tell when a message was printed.

3. No severity level. A debug message and a critical error look identical.

4. Only goes to terminal. You cannot redirect to a file without extra code.

5. No context. You cannot tell which module, class, or function generated the message.

**Logging solves all of these:**

1. Levels: set the minimum level once, all lower-level messages are suppressed automatically.
2. Timestamps: built into the format string.
3. Severity: DEBUG, INFO, WARNING, ERROR, CRITICAL.
4. Handlers: simultaneously write to console, file, database, network.
5. Context: `__name__` records the module automatically.

---

## The 5 Logging Levels

Levels are hierarchical. Setting a minimum level filters out everything below it.

```
Level      Numeric  When to use
-------    -------  -----------
DEBUG      10       Detailed diagnostic info. Variables, loop iterations.
                    Only needed when debugging a specific problem.
                    
INFO       20       Confirmation that things are working as expected.
                    "Training epoch 3/10 complete. Loss: 0.234"
                    
WARNING    30       Something unexpected happened but the program continues.
                    "Disk space below 10%. Results may not save."
                    
ERROR      40       A serious problem. A function failed to complete.
                    "NaN loss detected. Training unstable."
                    
CRITICAL   50       The program cannot continue. Immediate action needed.
                    "Database connection lost. Shutting down."
```

In development: set level to DEBUG (see everything).
In production: set level to INFO or WARNING (suppress debug noise).
When debugging a bug: temporarily set to DEBUG again.

In [14]:
# LOGGING BASICS: The 5 levels

# Remove any previous logging configuration (important in notebooks where cells re-run)
logging.getLogger().handlers.clear()

# basicConfig: simplest way to configure logging
# This MUST be called before any logging calls
# If you call logging.info() before basicConfig, Python uses default settings
logging.basicConfig(
    level=logging.DEBUG,         # minimum level: show DEBUG and above
    format='%(asctime)s | %(levelname)-8s | %(message)s',
    datefmt='%H:%M:%S'           # time format: hours:minutes:seconds
)

print('--- The 5 logging levels ---')
print()

logging.debug('This is DEBUG: variable x=42, loop iteration 7/100')
logging.info('This is INFO: model loaded successfully, 1000 samples processed')
logging.warning('This is WARNING: learning rate too high, loss oscillating')
logging.error('This is ERROR: file not found, batch processing failed')
logging.critical('This is CRITICAL: out of memory, cannot continue training')

11:58:28 | DEBUG    | This is DEBUG: variable x=42, loop iteration 7/100
11:58:28 | INFO     | This is INFO: model loaded successfully, 1000 samples processed
11:58:28 | WARNING  | This is WARNING: learning rate too high, loss oscillating
11:58:28 | ERROR    | This is ERROR: file not found, batch processing failed
11:58:28 | CRITICAL | This is CRITICAL: out of memory, cannot continue training


--- The 5 logging levels ---



In [15]:
# FORMAT STRINGS: What you can include in log messages

# The format string uses %(attribute)s syntax
# Common attributes:
#   %(asctime)s    - human-readable timestamp (controlled by datefmt)
#   %(created)f    - Unix timestamp (seconds since epoch)
#   %(levelname)s  - level as string: DEBUG, INFO, WARNING, ERROR, CRITICAL
#   %(levelno)d    - level as number: 10, 20, 30, 40, 50
#   %(name)s       - logger name (set by getLogger(name))
#   %(filename)s   - filename where logging call was made
#   %(funcName)s   - function name where logging call was made
#   %(lineno)d     - line number in source file
#   %(message)s    - the actual log message
#   %(process)d    - process ID
#   %(thread)d     - thread ID

logging.getLogger().handlers.clear()

logging.basicConfig(
    level=logging.DEBUG,
    format='%(asctime)s | %(name)-15s | %(levelname)-8s | %(filename)s:%(lineno)d | %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)

logging.info('Detailed format with file name and line number')
logging.warning('This shows which module and line generated the message')

2026-05-07 11:58:28 | root            | INFO     | 3269565710.py:25 | Detailed format with file name and line number
2026-05-07 11:58:28 | root            | WARNING  | 3269565710.py:26 | This shows which module and line generated the message


In [16]:
# LOGGER OBJECTS: The professional way

# logging.info() uses the ROOT logger - shared by all modules
# In production code, each module should have its OWN logger:
# logger = logging.getLogger(__name__)
#
# __name__ is a Python special variable that contains the module name
# In a file called train_model.py, __name__ is 'train_model'
# In a package called mypackage/utils.py, __name__ is 'mypackage.utils'
# This creates a hierarchy: mypackage -> mypackage.utils
#
# Why per-module loggers?
# You can set different log levels for different modules:
# - Your code: DEBUG level
# - Third-party libraries: WARNING level (suppress their verbose messages)

logging.getLogger().handlers.clear()

# Create module-specific loggers
logger_training = logging.getLogger('ml.training')   # simulating train_model.py
logger_data     = logging.getLogger('ml.data')       # simulating data_loader.py
logger_eval     = logging.getLogger('ml.eval')       # simulating evaluation.py

# Set up console handler with format
handler = logging.StreamHandler()
handler.setLevel(logging.DEBUG)
formatter = logging.Formatter('%(asctime)s | %(name)-14s | %(levelname)-8s | %(message)s',
                               datefmt='%H:%M:%S')
handler.setFormatter(formatter)

# Add handler to each logger
for logger_obj in [logger_training, logger_data, logger_eval]:
    logger_obj.addHandler(handler)
    logger_obj.setLevel(logging.DEBUG)
    logger_obj.propagate = False  # do not pass to root logger (avoids duplicate messages)

print('--- Named Loggers ---')
print()
logger_data.info('Loading dataset from disk...')
logger_data.info('Dataset loaded: 50000 samples, 128 features')
logger_training.info('Starting training for 10 epochs')
logger_training.debug('Batch 1/500: loss=2.3045')
logger_training.warning('Learning rate 0.1 may be too high for this dataset')
logger_eval.info('Epoch 1 complete: val_accuracy=0.734')
logger_training.error('NaN detected in loss at epoch 3 batch 47')

11:58:29 | ml.data        | INFO     | Loading dataset from disk...
11:58:29 | ml.data        | INFO     | Dataset loaded: 50000 samples, 128 features
11:58:29 | ml.training    | INFO     | Starting training for 10 epochs
11:58:29 | ml.training    | DEBUG    | Batch 1/500: loss=2.3045
11:58:29 | ml.training    | WARNING  | Learning rate 0.1 may be too high for this dataset
11:58:29 | ml.eval        | INFO     | Epoch 1 complete: val_accuracy=0.734
11:58:29 | ml.training    | ERROR    | NaN detected in loss at epoch 3 batch 47


--- Named Loggers ---



In [17]:
# HANDLERS: Multiple destinations simultaneously
# A handler determines WHERE log messages go
# You can add multiple handlers to one logger

# StreamHandler: writes to console (stdout or stderr)
# FileHandler: writes to a specific file
# RotatingFileHandler: writes to file, creates new file when size limit reached

import logging.handlers
import os

os.makedirs('day22_logs', exist_ok=True)

# Create a fresh logger for this demonstration
logger_demo = logging.getLogger('demo.handlers')
logger_demo.setLevel(logging.DEBUG)
logger_demo.propagate = False
logger_demo.handlers.clear()

# Format string used by all handlers
log_format = logging.Formatter(
    '%(asctime)s | %(name)s | %(levelname)-8s | %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)

# Handler 1: Console (StreamHandler)
# Shows INFO and above in the console (suppress DEBUG from console)
console_handler = logging.StreamHandler()
console_handler.setLevel(logging.INFO)    # console: INFO and above
console_handler.setFormatter(log_format)

# Handler 2: Regular FileHandler
# Writes everything (DEBUG and above) to a file
file_handler = logging.FileHandler(
    'day22_logs/app.log',
    mode='a',         # 'a' = append, 'w' = overwrite
    encoding='utf-8'
)
file_handler.setLevel(logging.DEBUG)      # file: DEBUG and above (everything)
file_handler.setFormatter(log_format)

# Handler 3: RotatingFileHandler
# maxBytes: rotate when file reaches this size (1MB here)
# backupCount: keep this many old files (rotating.log, rotating.log.1, rotating.log.2)
# When file reaches maxBytes, it is renamed to .log.1, and a new .log starts
# Old .log.1 becomes .log.2, etc. After backupCount files, oldest is deleted.
rotating_handler = logging.handlers.RotatingFileHandler(
    'day22_logs/rotating.log',
    maxBytes=1024*1024,   # 1 MB per file
    backupCount=3,         # keep last 3 rotated files
    encoding='utf-8'
)
rotating_handler.setLevel(logging.DEBUG)
rotating_handler.setFormatter(log_format)

# Add all handlers to the logger
logger_demo.addHandler(console_handler)
logger_demo.addHandler(file_handler)
logger_demo.addHandler(rotating_handler)

print('--- Multiple Handlers: Console + File + RotatingFile ---')
print()
logger_demo.debug('DEBUG: Only goes to file, not console (console level=INFO)')
logger_demo.info('INFO: Goes to BOTH console and file')
logger_demo.warning('WARNING: Goes to both console and file')
logger_demo.error('ERROR: Goes to both console and file')
print()
print('Check day22_logs/app.log - it will have DEBUG messages too')
print('Console shows only INFO+ because console_handler.setLevel(INFO)')

2026-05-07 11:58:30 | demo.handlers | INFO     | INFO: Goes to BOTH console and file
2026-05-07 11:58:30 | demo.handlers | WARNING  | WARNING: Goes to both console and file
2026-05-07 11:58:30 | demo.handlers | ERROR    | ERROR: Goes to both console and file


--- Multiple Handlers: Console + File + RotatingFile ---


Check day22_logs/app.log - it will have DEBUG messages too
Console shows only INFO+ because console_handler.setLevel(INFO)


In [18]:
# EXCEPTION LOGGING: How to log errors with full traceback

logger_exc = logging.getLogger('demo.exceptions')
logger_exc.setLevel(logging.DEBUG)
logger_exc.propagate = False
logger_exc.handlers.clear()
logger_exc.addHandler(console_handler)

print('--- Exception Logging ---')
print()

# Method 1: logger.exception() inside except block
# Automatically captures and formats the full traceback
# Use this ONLY inside except blocks
try:
    result = 10 / 0
except ZeroDivisionError:
    logger_exc.exception('Division failed - full traceback captured automatically')  
    # exception() = error() + exc_info=True (automatic)

print()

# Method 2: logger.error(msg, exc_info=True)
# Same result as exception() but can be used outside except if you have the exception
try:
    my_list = [1, 2, 3]
    x = my_list[10]   # IndexError
except IndexError:
    logger_exc.error('List index out of range', exc_info=True)

print()
print('logger.exception() vs logger.error(exc_info=True): same output')
print('exception() is shorthand for error(exc_info=True)')
print('Use exception() inside except blocks for cleaner code')

2026-05-07 11:58:30 | demo.exceptions | ERROR    | Division failed - full traceback captured automatically
Traceback (most recent call last):
  File "C:\Users\shaur\AppData\Local\Temp\ipykernel_1716\803093749.py", line 16, in <module>
    result = 10 / 0
             ~~~^~~
ZeroDivisionError: division by zero
2026-05-07 11:58:30 | demo.exceptions | ERROR    | List index out of range
Traceback (most recent call last):
  File "C:\Users\shaur\AppData\Local\Temp\ipykernel_1716\803093749.py", line 27, in <module>
    x = my_list[10]   # IndexError
        ~~~~~~~^^^^
IndexError: list index out of range


--- Exception Logging ---



logger.exception() vs logger.error(exc_info=True): same output
exception() is shorthand for error(exc_info=True)
Use exception() inside except blocks for cleaner code


In [19]:
# LOGGING DISABLE AND FILTER

print('--- Disabling Logging ---')
print()

logger_filter = logging.getLogger('demo.filter')
logger_filter.setLevel(logging.DEBUG)
logger_filter.propagate = False
logger_filter.handlers.clear()
logger_filter.addHandler(console_handler)

logger_filter.info('Message 1: visible')
logger_filter.info('Message 2: visible')

# Disable all logging below CRITICAL level
logging.disable(logging.WARNING)  # disables WARNING and below globally

logger_filter.info('Message 3: INVISIBLE (disabled below WARNING)')
logger_filter.warning('Message 4: INVISIBLE (disabled at WARNING level)')
logger_filter.error('Message 5: visible (ERROR > WARNING)')

# Re-enable all logging
logging.disable(logging.NOTSET)   # NOTSET = 0 = disable nothing

logger_filter.info('Message 6: visible again (disable reset)')
print()
print('logging.disable(level): silences all messages at or below that level')
print('logging.disable(logging.NOTSET): re-enables everything')
print()
print('Practical use: in unit tests, disable WARNING to suppress expected warnings')
print('In production scripts, disable DEBUG to quiet verbose library output')

2026-05-07 11:58:31 | demo.filter | INFO     | Message 1: visible
2026-05-07 11:58:31 | demo.filter | INFO     | Message 2: visible
2026-05-07 11:58:31 | demo.filter | ERROR    | Message 5: visible (ERROR > WARNING)
2026-05-07 11:58:31 | demo.filter | INFO     | Message 6: visible again (disable reset)


--- Disabling Logging ---


logging.disable(level): silences all messages at or below that level
logging.disable(logging.NOTSET): re-enables everything

Practical use: in unit tests, disable WARNING to suppress expected warnings
In production scripts, disable DEBUG to quiet verbose library output


---

# Part 5: Production ML Training Logger

## The Standard ML Logging Pattern

In real machine learning code (PyTorch, TensorFlow, scikit-learn), the logging follows a consistent pattern:

1. Logger is created once at module level: `logger = logging.getLogger(__name__)`
2. Two handlers: console for real-time monitoring, file for permanent record
3. Training loop logs INFO for normal progress
4. Errors (NaN, memory, I/O failures) logged at ERROR with exception info
5. Performance metrics (time per epoch, loss trajectory) included in messages
6. File uses RotatingFileHandler to prevent log files from filling up disk

This is the exact pattern you will see in production ML code at companies.

In [20]:
# PRODUCTION ML TRAINING LOGGER
# Simulates a real ML training loop with professional logging

import logging
import logging.handlers
import time
import random
import math
import os

os.makedirs('day22_logs', exist_ok=True)

def setup_ml_logger(name, log_file, console_level=logging.INFO, file_level=logging.DEBUG):
    """
    Create a professional logger with console + rotating file handlers.
    
    This function is the pattern used in real ML projects.
    Call it once at module startup, use the returned logger throughout.
    """
    logger = logging.getLogger(name)
    logger.setLevel(logging.DEBUG)  # logger accepts all; handlers filter
    logger.handlers.clear()         # prevent duplicate handlers in notebooks
    logger.propagate = False
    
    fmt = logging.Formatter(
        '%(asctime)s | %(name)s | %(levelname)-8s | %(message)s',
        datefmt='%H:%M:%S'
    )
    
    # Console handler: INFO and above
    ch = logging.StreamHandler()
    ch.setLevel(console_level)
    ch.setFormatter(fmt)
    logger.addHandler(ch)
    
    # Rotating file handler: DEBUG and above (everything)
    # maxBytes=5MB, keep 3 backup files
    fh = logging.handlers.RotatingFileHandler(
        log_file,
        maxBytes=5*1024*1024,
        backupCount=3,
        encoding='utf-8'
    )
    fh.setLevel(file_level)
    fh.setFormatter(fmt)
    logger.addHandler(fh)
    
    return logger


def simulate_training(n_epochs=10, inject_nan_at_epoch=4):
    """
    Simulates an ML training loop with realistic logging.
    Injects a NaN loss at inject_nan_at_epoch to demonstrate error logging.
    """
    logger = setup_ml_logger(
        name='ml.trainer',
        log_file='day22_logs/training.log'
    )
    
    random.seed(42)
    
    # Hyperparameters - log them at start
    config = {
        'epochs': n_epochs,
        'learning_rate': 0.001,
        'batch_size': 32,
        'model': 'SimpleCNN',
        'dataset': 'CIFAR-10'
    }
    
    logger.info('=' * 60)
    logger.info('Starting ML Training Session')
    logger.info('Configuration:')
    for k, v in config.items():
        logger.info(f'  {k}: {v}')
    logger.info('=' * 60)
    
    best_val_acc = 0.0
    training_start = time.perf_counter()
    
    for epoch in range(1, n_epochs + 1):
        epoch_start = time.perf_counter()
        
        # Simulate training metrics
        # Inject NaN at specified epoch to demonstrate error handling
        if epoch == inject_nan_at_epoch:
            train_loss = float('nan')
        else:
            # Realistic loss curve: starts high, decreases with noise
            base_loss = 2.5 * math.exp(-0.3 * epoch)
            train_loss = base_loss + random.gauss(0, 0.05)
        
        val_acc = min(0.95, 0.5 + 0.04 * epoch + random.gauss(0, 0.01))
        val_loss = train_loss * random.uniform(1.0, 1.2) if not math.isnan(train_loss) else float('nan')
        
        # Simulate epoch duration
        time.sleep(0.02)  # 20ms fake computation time
        epoch_time = time.perf_counter() - epoch_start
        
        # Check for NaN - critical error in training
        if math.isnan(train_loss):
            logger.error(
                f'Epoch {epoch:02d}/{n_epochs} | NaN detected in training loss. '
                f'Possible causes: exploding gradients, bad learning rate, corrupt data batch.'
            )
            logger.warning('Attempting to continue training... consider gradient clipping')
            # In real code you might: load best checkpoint, reduce LR, skip this batch
            continue  # skip this epoch
        
        # Check if this is the best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            logger.info(
                f'Epoch {epoch:02d}/{n_epochs} | '
                f'train_loss={train_loss:.4f} | '
                f'val_loss={val_loss:.4f} | '
                f'val_acc={val_acc:.4f} | '
                f'time={epoch_time:.3f}s | '
                f'[NEW BEST - checkpoint saved]'
            )
        else:
            logger.info(
                f'Epoch {epoch:02d}/{n_epochs} | '
                f'train_loss={train_loss:.4f} | '
                f'val_loss={val_loss:.4f} | '
                f'val_acc={val_acc:.4f} | '
                f'time={epoch_time:.3f}s'
            )
        
        # DEBUG: detailed batch-level info (only goes to file, not console)
        logger.debug(f'  Epoch {epoch} debug: lr=0.001, grad_norm=1.234, batch_count=1562')
        
        # Check for learning rate schedule (example: reduce after epoch 5)
        if epoch == 5:
            logger.warning('Epoch 5 reached. Reducing learning rate: 0.001 -> 0.0001')
    
    # Final summary
    total_time = time.perf_counter() - training_start
    logger.info('=' * 60)
    logger.info('Training Complete')
    logger.info(f'Best validation accuracy: {best_val_acc:.4f}')
    logger.info(f'Total training time: {total_time:.2f} seconds')
    logger.info(f'Average per epoch: {total_time/n_epochs:.3f} seconds')
    logger.info('=' * 60)
    
    return best_val_acc


# Run the simulation
print('Starting ML Training Simulation')
print('(Console shows INFO+. File contains DEBUG+ including batch details)')
print()
best_acc = simulate_training(n_epochs=10, inject_nan_at_epoch=4)
print(f'\nSimulation complete. Best accuracy: {best_acc:.4f}')
print('Check day22_logs/training.log for the full log including DEBUG messages')

11:58:32 | ml.trainer | INFO     | ============================================================
11:58:32 | ml.trainer | INFO     | Starting ML Training Session
11:58:32 | ml.trainer | INFO     | Configuration:
11:58:32 | ml.trainer | INFO     |   epochs: 10
11:58:32 | ml.trainer | INFO     |   learning_rate: 0.001
11:58:32 | ml.trainer | INFO     |   batch_size: 32
11:58:32 | ml.trainer | INFO     |   model: SimpleCNN
11:58:32 | ml.trainer | INFO     |   dataset: CIFAR-10
11:58:32 | ml.trainer | INFO     | ============================================================
11:58:32 | ml.trainer | INFO     | Epoch 01/10 | train_loss=1.8448 | val_loss=1.9463 | val_acc=0.5383 | time=0.020s | [NEW BEST - checkpoint saved]
11:58:32 | ml.trainer | INFO     | Epoch 02/10 | train_loss=1.3857 | val_loss=1.5733 | val_acc=0.5961 | time=0.021s | [NEW BEST - checkpoint saved]
11:58:32 | ml.trainer | INFO     | Epoch 03/10 | train_loss=1.0330 | val_loss=1.1202 | val_acc=0.6173 | time=0.020s | [NEW BEST - c

Starting ML Training Simulation
(Console shows INFO+. File contains DEBUG+ including batch details)



11:58:32 | ml.trainer | INFO     | Epoch 09/10 | train_loss=0.0953 | val_loss=0.1018 | val_acc=0.8740 | time=0.021s | [NEW BEST - checkpoint saved]
11:58:32 | ml.trainer | INFO     | Epoch 10/10 | train_loss=0.2285 | val_loss=0.2672 | val_acc=0.9038 | time=0.020s | [NEW BEST - checkpoint saved]
11:58:32 | ml.trainer | INFO     | ============================================================
11:58:32 | ml.trainer | INFO     | Training Complete
11:58:32 | ml.trainer | INFO     | Best validation accuracy: 0.9038
11:58:32 | ml.trainer | INFO     | Total training time: 0.22 seconds
11:58:32 | ml.trainer | INFO     | Average per epoch: 0.022 seconds
11:58:32 | ml.trainer | INFO     | ============================================================



Simulation complete. Best accuracy: 0.9038
Check day22_logs/training.log for the full log including DEBUG messages


In [21]:
# Read the log file to show what was saved
print('\nContents of day22_logs/training.log:')
print('-' * 60)
with open('day22_logs/training.log', 'r') as f:
    content = f.read()
print(content)
print('-' * 60)
print('Note: File contains DEBUG lines with batch details that were not shown in console')


Contents of day22_logs/training.log:
------------------------------------------------------------
11:58:32 | ml.trainer | INFO     | ============================================================
11:58:32 | ml.trainer | INFO     | Starting ML Training Session
11:58:32 | ml.trainer | INFO     | Configuration:
11:58:32 | ml.trainer | INFO     |   epochs: 10
11:58:32 | ml.trainer | INFO     |   learning_rate: 0.001
11:58:32 | ml.trainer | INFO     |   batch_size: 32
11:58:32 | ml.trainer | INFO     |   model: SimpleCNN
11:58:32 | ml.trainer | INFO     |   dataset: CIFAR-10
11:58:32 | ml.trainer | INFO     | ============================================================
11:58:32 | ml.trainer | INFO     | Epoch 01/10 | train_loss=1.8448 | val_loss=1.9463 | val_acc=0.5383 | time=0.020s | [NEW BEST - checkpoint saved]
11:58:32 | ml.trainer | DEBUG    |   Epoch 1 debug: lr=0.001, grad_norm=1.234, batch_count=1562
11:58:32 | ml.trainer | INFO     | Epoch 02/10 | train_loss=1.3857 | val_loss=1.5733

---

# GitHub File 1: math_random_demo.py - Monte Carlo Pi Estimator

In [22]:
# ============================================================
# day22_stdlib_part2/math_random_demo.py
# Monte Carlo Pi Estimator: math + random + time
# ============================================================

import math
import random
import time

def estimate_pi_monte_carlo(n_points: int, seed: int = 42) -> tuple:
    """
    Estimate pi using Monte Carlo method.

    Algorithm:
    1. Generate random points (x, y) in square [-1, 1] x [-1, 1]
    2. Count points inside unit circle: x^2 + y^2 <= 1
    3. pi/4 = inside_count / total_count
    4. pi = 4 * inside_count / total_count

    Returns:
        (pi_estimate: float, elapsed_time: float)
    """
    random.seed(seed)
    inside = 0
    start = time.perf_counter()

    for _ in range(n_points):
        x = random.uniform(-1.0, 1.0)
        y = random.uniform(-1.0, 1.0)
        if math.hypot(x, y) <= 1.0:
            inside += 1

    elapsed = time.perf_counter() - start
    pi_estimate = 4.0 * inside / n_points
    return pi_estimate, elapsed


def run_comparison():
    """Compare accuracy and speed across different sample sizes."""
    print('Monte Carlo Pi Estimation: Accuracy vs Sample Size')
    print(f'True pi = {math.pi}')
    print()
    print(f'{"N samples":>15}  {"Pi estimate":>12}  {"Abs error":>12}  {"Time (s)":>10}  {"Rel error":>12}')
    print('-' * 70)

    sample_sizes = [1_000, 10_000, 100_000, 1_000_000]

    for n in sample_sizes:
        estimate, t = estimate_pi_monte_carlo(n)
        abs_error = abs(estimate - math.pi)
        rel_error = abs_error / math.pi
        print(f'{n:>15,}  {estimate:>12.7f}  {abs_error:>12.7f}  {t:>10.4f}  {rel_error:>11.4%}')

    print()
    print('Key insight: error decreases with sqrt(N)')
    print('To halve error: need 4x more samples')
    print('To reduce error 10x: need 100x more samples')
    print()
    print('Mathematical background:')
    print(f'  circle area = pi * r^2 = pi * 1^2 = pi = {math.pi:.6f}')
    print(f'  square area = (2r)^2 = 4 = {4}')
    print(f'  ratio = pi/4 = {math.pi/4:.6f}')
    print(f'  pi = 4 * (inside/total)')
    print()
    print('random module note:')
    print('  random.uniform(-1, 1) generates uniform float in [-1, 1]')
    print('  math.hypot(x, y) = sqrt(x^2 + y^2): Euclidean distance from origin')
    print('  time.perf_counter(): high-resolution timer for performance measurement')


def demonstrate_math_functions():
    """Showcase key math module functions used in data science."""
    print('\nmath Module Key Functions:')
    print('-' * 40)

    print(f'Constants:  pi={math.pi:.6f}, e={math.e:.6f}, tau={math.tau:.6f}')
    print(f'sqrt(2)   = {math.sqrt(2):.6f}')
    print(f'log2(1024) = {math.log2(1024)}')
    print(f'log10(1e6) = {math.log10(1e6)}')
    print(f'exp(1)    = {math.exp(1):.6f}  (= e)')
    print(f'sin(pi/2) = {math.sin(math.pi/2)}')
    print(f'comb(10,3) = {math.comb(10, 3)}  (10 choose 3)')
    print(f'perm(10,3) = {math.perm(10, 3)}  (10 arrange 3)')
    print(f'factorial(10) = {math.factorial(10)}')
    print(f'gcd(48, 18)  = {math.gcd(48, 18)}')
    print(f'isclose(0.1+0.2, 0.3) = {math.isclose(0.1+0.2, 0.3)}')


run_comparison()
demonstrate_math_functions()

Monte Carlo Pi Estimation: Accuracy vs Sample Size
True pi = 3.141592653589793

      N samples   Pi estimate     Abs error    Time (s)     Rel error
----------------------------------------------------------------------
          1,000     3.1800000     0.0384073      0.0005      1.2225%
         10,000     3.1392000     0.0023927      0.0045      0.0762%
        100,000     3.1402800     0.0013127      0.0422      0.0418%
      1,000,000     3.1402440     0.0013487      0.3413      0.0429%

Key insight: error decreases with sqrt(N)
To halve error: need 4x more samples
To reduce error 10x: need 100x more samples

Mathematical background:
  circle area = pi * r^2 = pi * 1^2 = pi = 3.141593
  square area = (2r)^2 = 4 = 4
  ratio = pi/4 = 0.785398
  pi = 4 * (inside/total)

random module note:
  random.uniform(-1, 1) generates uniform float in [-1, 1]
  math.hypot(x, y) = sqrt(x^2 + y^2): Euclidean distance from origin
  time.perf_counter(): high-resolution timer for performance measurem

---

# GitHub File 2: production_logger.py - ML Training Logger

In [23]:
# ============================================================
# day22_stdlib_part2/production_logger.py
# ML Training Logger: logging + time + random + math
# ============================================================

import logging
import logging.handlers
import time
import random
import math
import os

# Module-level logger - the standard pattern
# In a real file, __name__ = 'production_logger'
logger = logging.getLogger('ml.trainer')


def setup_logging(log_dir: str = 'day22_logs') -> None:
    """
    Configure logging with console and rotating file handlers.
    Call this ONCE at application startup.
    """
    os.makedirs(log_dir, exist_ok=True)

    logger.setLevel(logging.DEBUG)
    logger.handlers.clear()
    logger.propagate = False

    fmt = logging.Formatter(
        '%(asctime)s | %(name)-12s | %(levelname)-8s | %(message)s',
        datefmt='%Y-%m-%d %H:%M:%S'
    )

    # Console: INFO and above
    ch = logging.StreamHandler()
    ch.setLevel(logging.INFO)
    ch.setFormatter(fmt)
    logger.addHandler(ch)

    # File: DEBUG and above (full history)
    fh = logging.handlers.RotatingFileHandler(
        os.path.join(log_dir, 'ml_training.log'),
        maxBytes=5 * 1024 * 1024,  # 5 MB
        backupCount=5,
        encoding='utf-8'
    )
    fh.setLevel(logging.DEBUG)
    fh.setFormatter(fmt)
    logger.addHandler(fh)

    logger.info('Logging initialized. Console=INFO, File=DEBUG')
    logger.info(f'Log file: {os.path.join(log_dir, "ml_training.log")}')


def train_model(n_epochs: int = 10) -> dict:
    """
    Fake ML training loop demonstrating production logging patterns.
    Injects NaN at epoch 4 to demonstrate error handling.
    """
    random.seed(42)

    config = {
        'epochs': n_epochs,
        'learning_rate': 0.001,
        'batch_size': 64,
        'model_type': 'ResNet-18',
        'dataset': 'ImageNet-subset'
    }

    logger.info('=' * 55)
    logger.info('NEW TRAINING RUN STARTED')
    for k, v in config.items():
        logger.info(f'  {k:<18}: {v}')
    logger.info('=' * 55)

    best_val_acc = 0.0
    history = []
    session_start = time.perf_counter()
    nan_count = 0

    for epoch in range(1, n_epochs + 1):
        epoch_start = time.perf_counter()

        # Simulate metrics
        if epoch == 4:
            train_loss = math.nan  # inject NaN
        else:
            train_loss = 2.3 * math.exp(-0.25 * epoch) + random.gauss(0, 0.04)
            train_loss = max(0.05, train_loss)  # floor at 0.05

        val_acc = min(0.96, 0.45 + 0.045 * epoch + random.gauss(0, 0.008))

        # Simulate batch processing time
        time.sleep(0.015)
        epoch_sec = time.perf_counter() - epoch_start

        # NaN detection - critical error handling
        if math.isnan(train_loss) or math.isinf(train_loss):
            nan_count += 1
            logger.error(
                f'[Epoch {epoch:02d}] NaN/Inf in training loss. '
                f'Possible: exploding gradients, bad LR, corrupt batch. '
                f'NaN count this session: {nan_count}'
            )
            if nan_count >= 3:
                logger.critical('3 consecutive NaN losses. Stopping training.')
                break
            logger.warning('Skipping epoch and continuing...')
            continue

        # New best model
        is_best = val_acc > best_val_acc
        if is_best:
            best_val_acc = val_acc

        status = '[BEST]' if is_best else ''
        logger.info(
            f'Epoch {epoch:02d}/{n_epochs} | '
            f'loss={train_loss:.4f} | '
            f'val_acc={val_acc:.4f} | '
            f'{epoch_sec:.3f}s {status}'
        )

        # DEBUG: detailed per-epoch diagnostics (file only)
        logger.debug(
            f'Epoch {epoch} internals: '
            f'lr=0.001, grad_norm={random.uniform(0.1, 2.0):.4f}, '
            f'batches=1250, samples/sec={random.randint(800, 1200)}'
        )

        # LR schedule warning
        if epoch == 7:
            logger.warning('Plateau detected. Reducing LR: 0.001 -> 0.0001')

        history.append({
            'epoch': epoch,
            'loss': round(train_loss, 4),
            'val_acc': round(val_acc, 4),
            'time': round(epoch_sec, 3)
        })

    # Final summary
    total_time = time.perf_counter() - session_start
    logger.info('=' * 55)
    logger.info('TRAINING COMPLETE')
    logger.info(f'Best validation accuracy : {best_val_acc:.4f}')
    logger.info(f'Total time              : {total_time:.2f}s')
    logger.info(f'Epochs completed        : {len(history)}/{n_epochs}')
    logger.info(f'NaN epochs skipped      : {nan_count}')
    logger.info('=' * 55)

    return {'best_val_acc': best_val_acc, 'history': history}


# Run
setup_logging(log_dir='day22_logs')
results = train_model(n_epochs=10)
print(f'\nFinal best accuracy: {results["best_val_acc"]:.4f}')
print(f'Epochs in history: {len(results["history"])}')

2026-05-07 11:58:35 | ml.trainer   | INFO     | Logging initialized. Console=INFO, File=DEBUG
2026-05-07 11:58:35 | ml.trainer   | INFO     | Log file: day22_logs\ml_training.log
2026-05-07 11:58:35 | ml.trainer   | INFO     | =======================================================
2026-05-07 11:58:35 | ml.trainer   | INFO     | NEW TRAINING RUN STARTED
2026-05-07 11:58:35 | ml.trainer   | INFO     |   epochs            : 10
2026-05-07 11:58:35 | ml.trainer   | INFO     |   learning_rate     : 0.001
2026-05-07 11:58:35 | ml.trainer   | INFO     |   batch_size        : 64
2026-05-07 11:58:35 | ml.trainer   | INFO     |   model_type        : ResNet-18
2026-05-07 11:58:35 | ml.trainer   | INFO     |   dataset           : ImageNet-subset
2026-05-07 11:58:35 | ml.trainer   | INFO     | =======================================================
2026-05-07 11:58:36 | ml.trainer   | INFO     | Epoch 01/10 | loss=1.7855 | val_acc=0.4936 | 0.016s [BEST]
2026-05-07 11:58:36 | ml.trainer   | INFO    


Final best accuracy: 0.9045
Epochs in history: 9


---

# Complete Reference: Day 22 Cheat Sheet

---

## math Module Quick Reference

```python
import math

# Constants
math.pi          # 3.141592653589793
math.e           # 2.718281828459045
math.tau         # 6.283185307179586 (= 2*pi)
math.inf         # infinity
math.nan         # Not a Number

# Basic arithmetic
math.sqrt(x)     # square root (always float)
math.pow(x, y)   # x^y (always float)
math.fabs(x)     # absolute value (always float)
math.factorial(n)# n!
math.gcd(a, b)   # greatest common divisor
math.lcm(a, b)   # least common multiple
math.floor(x)    # round down toward -infinity
math.ceil(x)     # round up toward +infinity
math.trunc(x)    # remove decimal toward zero

# Logarithms and exponentials
math.log(x)      # natural log (base e)
math.log(x, b)   # log base b
math.log2(x)     # log base 2 (more accurate than log(x,2))
math.log10(x)    # log base 10 (more accurate than log(x,10))
math.exp(x)      # e^x
math.log1p(x)    # log(1+x): stable for small x
math.expm1(x)    # e^x - 1: stable for small x

# Trigonometry (all angles in RADIANS)
math.sin(x)      # sine
math.cos(x)      # cosine
math.tan(x)      # tangent
math.asin(x)     # inverse sine (arcsin)
math.acos(x)     # inverse cosine
math.atan(x)     # inverse tangent
math.atan2(y, x) # angle of point (x,y) - handles all quadrants
math.degrees(r)  # radians to degrees
math.radians(d)  # degrees to radians
math.hypot(x, y) # sqrt(x^2 + y^2): safe Pythagorean distance

# Validation
math.isclose(a, b, rel_tol=1e-9)  # float comparison (use instead of ==)
math.isnan(x)    # check if NaN (do NOT use x == nan)
math.isinf(x)    # check if infinity
math.isfinite(x) # check if normal number (not nan or inf)
math.fmod(x, y)  # float modulo (sign matches x, not y)

# Combinatorics
math.comb(n, k)  # C(n,k): combinations, order does not matter
math.perm(n, k)  # P(n,k): permutations, order matters
math.prod(iter)  # product of all elements
math.fsum(iter)  # accurate sum (avoids floating point accumulation)
```

---

## random Module Quick Reference

```python
import random

# Reproducibility (ALWAYS set seed in ML experiments)
random.seed(42)                   # fixed seed = reproducible sequence

# Float generators
random.random()                   # float in [0.0, 1.0)
random.uniform(a, b)              # float in [a, b]

# Integer generators
random.randint(a, b)              # int in [a, b] - BOTH endpoints included
random.randrange(start, stop, step)# int in [start, stop) - stop NOT included

# Distributions
random.gauss(mu, sigma)           # Gaussian/Normal (faster, not thread-safe)
random.normalvariate(mu, sigma)   # Gaussian/Normal (thread-safe)
random.expovariate(lambd)         # Exponential (lambd = 1/mean)

# Sampling
random.choice(seq)                # pick ONE element
random.choices(seq, weights, k)   # pick k WITH replacement
random.sample(seq, k)             # pick k WITHOUT replacement (no duplicates)
random.shuffle(seq)               # shuffle IN-PLACE, returns None

# choices vs sample:
# choices: can repeat (like rolling dice)
# sample:  no repeats (like dealing cards)
```

---

## logging Module Quick Reference

```python
import logging
import logging.handlers

# Level hierarchy (each level includes all above it):
# DEBUG=10, INFO=20, WARNING=30, ERROR=40, CRITICAL=50

# Simple setup (for scripts)
logging.basicConfig(
    level=logging.DEBUG,
    format='%(asctime)s | %(levelname)-8s | %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S',
    filename='app.log',   # omit for console output
    filemode='a'          # 'a'=append, 'w'=overwrite
)

# Professional pattern (for modules and packages)
logger = logging.getLogger(__name__)   # __name__ = module name
logger.setLevel(logging.DEBUG)

# Handlers
ch = logging.StreamHandler()           # console
ch.setLevel(logging.INFO)

fh = logging.FileHandler('app.log')   # file
fh.setLevel(logging.DEBUG)

rfh = logging.handlers.RotatingFileHandler(
    'app.log',
    maxBytes=5*1024*1024,   # 5 MB
    backupCount=3
)

# Formatter
fmt = logging.Formatter(
    '%(asctime)s | %(name)s | %(levelname)-8s | %(message)s',
    datefmt='%H:%M:%S'
)
ch.setFormatter(fmt)
fh.setFormatter(fmt)

# Add handlers
logger.addHandler(ch)
logger.addHandler(fh)
logger.propagate = False   # prevent double-printing if root has handlers

# Logging calls
logger.debug('Variable x=%d', x)       # lazy formatting (preferred)
logger.info('Epoch %d/%d complete', e, n)
logger.warning('LR too high: %f', lr)
logger.error('Batch %d failed', batch_id)
logger.critical('Out of memory')

# Exception logging (inside except block)
try:
    result = risky_operation()
except Exception:
    logger.exception('Operation failed')   # logs ERROR + full traceback
    # OR:
    logger.error('Operation failed', exc_info=True)  # same

# Disable/enable
logging.disable(logging.WARNING)   # silence WARNING and below
logging.disable(logging.NOTSET)    # re-enable everything

# Format string attributes:
# %(asctime)s   - timestamp
# %(name)s      - logger name
# %(levelname)s - level string
# %(filename)s  - source file
# %(funcName)s  - function name
# %(lineno)d    - line number
# %(message)s   - the message
```

---

## The 3 Critical Habits

```
1. Never use == with floats
   Wrong:  if result == 0.3:
   Right:  if math.isclose(result, 0.3):

2. Never use random for security
   Wrong:  token = str(random.randint(100000, 999999))  # predictable
   Right:  token = secrets.token_urlsafe(16)            # cryptographically secure

3. Never use print() in production code
   Wrong:  print(f'Epoch {e}: loss={loss:.4f}')
   Right:  logger.info('Epoch %d: loss=%.4f', e, loss)
```

---